# SRV cx_level Relabelling

Extends the existing `qc_srv_dataset_3to8qubit` labels with a `; cx_level=low/medium/high` suffix
based on per-SRV tertiles of `cx_ratio = n_cx / n_gates`.

Output: a new dataset at `artifacts/datasets/srv-noise-conditioning/` that can be used
to fine-tune the SRV baseline model for CX-aware circuit generation.

In [ ]:
from shared.bootstrap import setup_notebook_paths
PROJECT_ROOT = setup_notebook_paths()

import torch
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path
import shutil
import yaml

from shared.evaluation_artifacts import save_figure

plt.rcParams.update({
    "figure.dpi": 150, "figure.facecolor": "white",
    "axes.facecolor": "white", "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlepad": 10,
    "axes.labelsize": 11, "axes.labelpad": 6,
    "axes.grid": True, "grid.linestyle": "--", "grid.linewidth": 0.6,
    "grid.alpha": 0.5, "grid.color": "#AAAAAA", "axes.axisbelow": True,
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "legend.fontsize": 10, "legend.framealpha": 0.92, "legend.edgecolor": "#CCCCCC",
    "lines.linewidth": 2.0, "lines.markersize": 6, "font.family": "sans-serif",
})

In [ ]:
# --- Config ---
SOURCE_DATASET = PROJECT_ROOT / "artifacts/datasets/qc_srv_dataset_3to8qubit"
OUTPUT_DATASET = PROJECT_ROOT / "artifacts/datasets/srv-noise-conditioning"

ARTIFACT_DIR   = PROJECT_ROOT / "artifacts/evaluations/srv-noise-conditioning/relabelling"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# --- Load source dataset ---
ds_x = torch.load(SOURCE_DATASET / "dataset/ds_x.pt", weights_only=False)
ds_y = torch.load(SOURCE_DATASET / "dataset/ds_y.pt", weights_only=False)
ds_z = torch.load(SOURCE_DATASET / "dataset/ds_z.pt", weights_only=False)

circuits = ds_x["0"]   # [N, 8, 52]  gate ids
labels   = ds_y["0"]   # [N]          CLIP text labels
z        = ds_z["0"]   # [N, 2]       (n_qubits, n_gates)

N = len(circuits)
print(f"Loaded {N:,} circuits")
print(f"Circuit tensor: {circuits.shape}, dtype={circuits.dtype}")
print(f"Sample label: {labels[0]}")
print(f"Sample z: {z[0].tolist()}")

In [ ]:
# --- Compute n_cx and cx_ratio per circuit ---
# CX gate: control qubit row = -2, target qubit row = 2.
# Count CX gates = number of time steps (columns) where any qubit has -2.
n_cx    = (circuits == -2).any(dim=1).sum(dim=1).cpu().numpy().astype(float)  # [N]
n_gates = z[:, 1].cpu().numpy().astype(float)                                  # [N]  actual depth

# cx_ratio: CX density — normalised for circuit length
cx_ratio = np.where(n_gates > 0, n_cx / n_gates, 0.0)                         # [N]

print(f"n_cx   — mean={n_cx.mean():.2f}, std={n_cx.std():.2f}, min={n_cx.min():.0f}, max={n_cx.max():.0f}")
print(f"n_gates— mean={n_gates.mean():.2f}, std={n_gates.std():.2f}")
print(f"cx_ratio — mean={cx_ratio.mean():.3f}, std={cx_ratio.std():.3f}, "
      f"min={cx_ratio.min():.3f}, max={cx_ratio.max():.3f}")

In [ ]:
# --- Per-SRV tertile thresholds of cx_ratio ---
# "low"    = bottom third of cx_ratio within this SRV
# "medium" = middle third
# "high"   = top third

srv_thresholds = {}  # srv_str -> (t33, t67)
for srv in np.unique(labels):
    mask = labels == srv
    ratios = cx_ratio[mask]
    srv_thresholds[srv] = np.percentile(ratios, [33.0, 67.0])

print(f"Computed thresholds for {len(srv_thresholds)} unique SRVs")

# Sanity: show a few
for srv, (t33, t67) in sorted(srv_thresholds.items())[:5]:
    print(f"  {srv}  →  low≤{t33:.3f}, medium≤{t67:.3f}")

In [ ]:
# --- Relabel ---
# Use object dtype to avoid fixed-width string truncation ("medium" > original label length)
new_labels = np.empty(len(labels), dtype=object)

for srv, (t33, t67) in srv_thresholds.items():
    mask  = labels == srv
    idxs  = np.where(mask)[0]
    for i, r in zip(idxs, cx_ratio[mask]):
        lvl = "low" if r <= t33 else ("medium" if r <= t67 else "high")
        new_labels[i] = f"{srv}; cx_level={lvl}"

print("Sample new labels:")
for lbl in new_labels[:6]:
    print(" ", lbl)

# Verify balance
from collections import Counter
level_counts = Counter(lbl.split("cx_level=")[-1] for lbl in new_labels)
total = sum(level_counts.values())
print("\nGlobal level balance:")
for lvl in ["low", "medium", "high"]:
    print(f"  {lvl}: {level_counts[lvl]:,}  ({100*level_counts[lvl]/total:.1f}%)")

In [ ]:
# --- Verification: cx_ratio distributions per level ---
level_ratios = {"low": [], "medium": [], "high": []}
for i, lbl in enumerate(new_labels):
    lvl = lbl.split("cx_level=")[-1]
    level_ratios[lvl].append(cx_ratio[i])

COLORS = {"low": "#2176AE", "medium": "#3BAA6E", "high": "#E05C2A"}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

# Left: cx_ratio histogram per level
ax = axes[0]
bins = np.linspace(0, 1, 41)
for lvl in ["low", "medium", "high"]:
    vals = np.array(level_ratios[lvl])
    weights = np.full(len(vals), 100.0 / len(vals))
    ax.hist(vals, bins=bins, weights=weights, alpha=0.65,
            color=COLORS[lvl], label=lvl, edgecolor="white", linewidth=0.4)
ax.set_xlabel("cx_ratio  (n_cx / n_gates)")
ax.set_ylabel("% of circuits in level")
ax.set_title("cx_ratio Distribution per Level")
ax.legend()
ax.yaxis.grid(True); ax.xaxis.grid(False)

# Right: median cx_ratio per SRV, coloured by qubit count
ax = axes[1]
q_vals = sorted(set(z[:, 0].cpu().numpy()))
cmap = plt.get_cmap("Oranges", len(q_vals))
for qi, q in enumerate(q_vals):
    q_mask = z[:, 0].cpu().numpy() == q
    for srv in np.unique(labels[q_mask]):
        srv_mask = (labels == srv) & q_mask
        low_m  = np.median([cx_ratio[i] for i, lbl in enumerate(new_labels) if srv_mask[i] and "cx_level=low"    in lbl])
        high_m = np.median([cx_ratio[i] for i, lbl in enumerate(new_labels) if srv_mask[i] and "cx_level=high"   in lbl])
        ax.scatter(low_m, high_m, color=cmap(qi), s=18, alpha=0.7)

ax.plot([0, 1], [0, 1], "--", color="#AAAAAA", linewidth=1)
ax.set_xlabel("Median cx_ratio — low bin")
ax.set_ylabel("Median cx_ratio — high bin")
ax.set_title("Per-SRV: low vs high Separation")

sm = mpl.cm.ScalarMappable(cmap=cmap, norm=mpl.colors.BoundaryNorm(
    np.arange(min(q_vals) - 0.5, max(q_vals) + 1.5), cmap.N))
plt.colorbar(sm, ax=ax, label="n_qubits", fraction=0.046, pad=0.04)

fig.tight_layout()
save_figure(fig, ARTIFACT_DIR / "cx_level_verification.png")
plt.show()

In [ ]:
# --- Save new dataset ---
out_ds = OUTPUT_DATASET / "dataset"
out_ds.mkdir(parents=True, exist_ok=True)

# ds_x and ds_z unchanged — copy
shutil.copy(SOURCE_DATASET / "dataset/ds_x.pt", out_ds / "ds_x.pt")
shutil.copy(SOURCE_DATASET / "dataset/ds_z.pt", out_ds / "ds_z.pt")

# ds_y — save with new labels
torch.save({"0": new_labels}, out_ds / "ds_y.pt")

# config.yaml — copy source, update comment and save_path
with open(SOURCE_DATASET / "config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["comment"] = (
    "Derived from qc_srv_dataset_3to8qubit. Labels extended with "
    "; cx_level=low/medium/high based on per-SRV tertiles of cx_ratio=n_cx/n_gates."
)
cfg["save_path"] = str(out_ds / "ds")

with open(OUTPUT_DATASET / "config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"Saved to {OUTPUT_DATASET}")
print(f"  ds_x.pt: {(out_ds / 'ds_x.pt').stat().st_size / 1e9:.2f} GB")
print(f"  ds_y.pt: {(out_ds / 'ds_y.pt').stat().st_size / 1e6:.1f} MB")
print(f"  ds_z.pt: {(out_ds / 'ds_z.pt').stat().st_size / 1e6:.1f} MB")